# 🎓 DegreeDetailExtract — v2 Anti-Overfitting Training Notebook

**What's new in v2** (vs the original notebook):
- 🖼️ Real paper textures extracted from `real_certs/` are blended into every generated certificate background
- 📝 Full ceremonial prose text (12 templates: "We, the President of Council...", "The Board of Management hereby certifies...", etc.) is rendered **inside every image**
- 🎨 28 total layout templates (22 original + 6 new prose+texture templates)
- 📦 3 000 images instead of 5 000 (diversity > volume)
- ⚡ 4 training epochs instead of 8 (reduces overfitting, saves ~45 min)
- 📸 Heavier augmentation simulating phone-camera certificate photos

**Pipeline**: Env Setup → Repo Clone + Real Cert Download → Generate 3k v2 Dataset → Zip & Save → Fine-tune Donut → Evaluate → Inference

**Target runtime on free Colab T4: ~90–100 minutes total**

---
Run sections **in order**. Each section header tells you what it does.


## 🛠️ Section 0 — Environment Setup

Installs all packages, verifies the GPU, and mounts Google Drive.

**Expected time: ~3 minutes**


In [ ]:
# Install system fonts and Python packages
!apt-get install -y -q fonts-liberation2 fonts-dejavu-core libgl1

!pip install -q \
    'transformers>=4.35.0' \
    'datasets>=2.14.0' \
    'accelerate>=0.24.0' \
    sentencepiece \
    Faker \
    'Pillow>=10.0.0' \
    opencv-python-headless \
    'albumentations>=1.3.0' \
    tqdm \
    'scikit-learn>=1.3.0' \
    editdistance \
    matplotlib \
    kaggle

print('\n✅ All packages installed.')

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'CUDA ver: {torch.version.cuda}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {mem_gb:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice  : {DEVICE}')

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH   = '/content/drive/MyDrive/DegreeDetailExtract'
CKPT_PATH    = f'{DRIVE_PATH}/checkpoints_v2'

for d in [DRIVE_PATH, CKPT_PATH]:
    os.makedirs(d, exist_ok=True)

print(f'Drive root   : {DRIVE_PATH}')
print(f'Checkpoints  : {CKPT_PATH}')

## 🖼️ Section 1 — Prepare Repository & Real Certificate Images

### Step 1a: Clone the GitHub repository
The repo already contains the `real_certs/` folder with your real certificate scans.

### Step 1b: Download additional real certificate images
We sample a small number of images from public datasets to increase texture variety
(we only download ~50–80 images, not the entire dataset).

**Expected time: ~2 minutes**


In [ ]:
import os, sys

REPO_URL = 'https://github.com/Vrushti33/DegreeDetailExtract.git'
REPO_DIR = '/content/DegreeDetailExtract'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}

real_certs = [f for f in os.listdir(f'{REPO_DIR}/real_certs') if not f.startswith('.')]
print(f'\n✅ Repository ready. Real certs in repo: {len(real_certs)}')
for f in real_certs:
    print(f'   {f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Download ~40 extra real certificate images from Roboflow public datasets.
# We intentionally cap at ~40 images (not the full dataset) to keep Colab
# storage under control while still giving the texture extractor variety.
#
# These certificates are used ONLY as background texture sources — no text
# from them appears in the generated training data.
# ─────────────────────────────────────────────────────────────────────────────

import os, urllib.request, random
from pathlib import Path

EXTRA_DIR = Path(f'{REPO_DIR}/real_certs')
EXTRA_DIR.mkdir(exist_ok=True)

# ── Roboflow export links (no API key needed for public datasets) ─────────────
# University Certificates dataset — first page of images
ROBOFLOW_DATASETS = [
    # Format: (base_url, image_name, filename_to_save)
    # We pull images directly from Roboflow's CDN via their export URLs.
    # These are the first ~20 images from the two public university-cert datasets.
    ('https://storage.googleapis.com/roboflow-platform-uploads',
     None,  # placeholder — see download function
     None),
]

# ── Alternative: use Roboflow pip SDK (more reliable) ─────────────────────────
def try_roboflow_download():
    """Try downloading via roboflow SDK. Returns True on success."""
    try:
        !pip install -q roboflow
        from roboflow import Roboflow

        # Dataset 1: university-certificates-uhuct
        rf = Roboflow(api_key="")  # public datasets don't need API key
        # This call may fail without key — handled by except below
        return False
    except Exception:
        return False

# ── Download from OpenImages-style direct CDN ─────────────────────────────────
# Since Roboflow requires an API key, we use a direct image archive from their
# public export. The URL below is the raw Roboflow ZIP for the public dataset.
ROBOFLOW_ZIP_URL = (
    'https://universe.roboflow.com/ds/3Ql1yl3dSh'
    '?key=IbSJGn8GR8'   # public API key from Roboflow universe (read-only)
)

def download_roboflow_sample():
    """Download a small sample of university certificate images."""
    import zipfile, io
    try:
        print('Attempting Roboflow dataset download...')
        req = urllib.request.Request(
            ROBOFLOW_ZIP_URL,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = resp.read()
        zf = zipfile.ZipFile(io.BytesIO(data))
        images = [n for n in zf.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png'))]
        # Extract only first 40 images
        count = 0
        for name in images[:40]:
            dest = EXTRA_DIR / f'roboflow_{count:03d}{Path(name).suffix}'
            dest.write_bytes(zf.read(name))
            count += 1
        print(f'Downloaded {count} extra real certificate images from Roboflow.')
        return count
    except Exception as e:
        print(f'Roboflow download skipped ({e}) — using only repo images.')
        return 0

extra_count = download_roboflow_sample()

all_real = list(EXTRA_DIR.iterdir())
print(f'\nTotal real certificate images available for texture extraction: {len(all_real)}')

## 🏭 Section 2 — Generate 3 000 Realistic v2 Certificates

This section:
1. Inpaints real scans to extract paper textures (cached in memory)
2. Generates 3 000 certificates with ceremonial prose + real backgrounds
3. Zips the dataset and saves to Google Drive

**Expected time: ~12–18 minutes**


In [ ]:
# ── Generate 3 000 v2 certificates ────────────────────────────────────────────
# --real_certs_dir  tells the generator where to find real images for textures
# --count 3000      fewer images → faster + more diversity per image
# --seed 42         reproducible

!python generate_certificates_v2.py \
    --count 3000 \
    --output_dir /content/dataset_v2 \
    --real_certs_dir /content/DegreeDetailExtract/real_certs \
    --seed 42 \
    --quality 90

print('\nGeneration complete.')

In [ ]:
# Preview 6 random v2 certificates to verify prose text + real backgrounds
import json, random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

IMG_DIR = Path('/content/dataset_v2/images')
files   = sorted(IMG_DIR.glob('*.jpg'))
print(f'Total images: {len(files)}')

samples = random.sample(files, min(6, len(files)))
fig, axes = plt.subplots(2, 3, figsize=(22, 18))
for ax, fpath in zip(axes.flat, samples):
    ax.imshow(Image.open(fpath))
    ax.set_title(fpath.name, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample v2 Certificates (prose + real texture)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Verify metadata
with open('/content/dataset_v2/metadata_train.jsonl') as f:
    records = [json.loads(l) for l in f]
print(f'\nTrain records: {len(records)}')
print('Sample record:')
print(json.dumps(records[0], indent=2))

In [ ]:
import shutil, os

# Compress the dataset — 1 file is far faster to copy to Drive than 3 000 files
print('Compressing dataset (1–2 min) ...')
zip_src = '/content/dataset_v2_archive'
shutil.make_archive(zip_src, 'zip', '/content', 'dataset_v2')
zip_gb  = os.path.getsize(zip_src + '.zip') / 1e9
print(f'Zip created: {zip_gb:.2f} GB')

dest_zip = f'{DRIVE_PATH}/dataset_v2_archive.zip'
print(f'Uploading to Drive: {dest_zip} ...')
shutil.copy(zip_src + '.zip', dest_zip)
print('Dataset zip saved to Drive.')

for split in ('train', 'val', 'test'):
    src = f'/content/dataset_v2/metadata_{split}.jsonl'
    dst = f'{DRIVE_PATH}/metadata_v2_{split}.jsonl'
    shutil.copy(src, dst)
    n = sum(1 for _ in open(src))
    print(f'  {split}: {n:,} records saved to Drive.')

## 🔥 Section 3 — Fine-tune Donut on v2 Dataset

**Key changes vs Round 1:**
- `num_train_epochs=4` (was 8) — reduces overfitting, saves ~45 minutes
- Checkpoints every **500 steps** (was 1000) — protects against Colab disconnects
- Same Adafactor + fp16 + gradient checkpointing for T4 memory safety

**Expected time: ~60–75 minutes on free T4**

> ⚠️ **If Colab disconnects mid-training**, re-run Section 0 cells, then run
> the *Resume from checkpoint* cell (below the training cell) instead of the
> main training cell.


In [ ]:
import gc, torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

# Free any previous model from memory
try:
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print('Cleared previous model from VRAM.')
except NameError:
    pass

print('Loading donut-base ...')
processor = DonutProcessor.from_pretrained('naver-clova-ix/donut-base')
model     = VisionEncoderDecoderModel.from_pretrained('naver-clova-ix/donut-base')

# Reduce resolution: 1280×960 → 640×480 (4× fewer attention ops)
IMG_H, IMG_W = 640, 480
processor.image_processor.size = {"height": IMG_H, "width": IMG_W}
model.config.encoder.image_size = [IMG_H, IMG_W]
print(f'Image size set to {IMG_H}×{IMG_W}')

# Add the 16 special tokens for 7 certificate fields
CERT_SPECIAL_TOKENS = [
    '<s_cert>', '</s_cert>',
    '<s_student_name>',    '</s_student_name>',
    '<s_university_name>', '</s_university_name>',
    '<s_course_name>',     '</s_course_name>',
    '<s_specialization>',  '</s_specialization>',
    '<s_pass_class>',      '</s_pass_class>',
    '<s_authority_name>',  '</s_authority_name>',
    '<s_issue_date>',      '</s_issue_date>',
]

processor.tokenizer.add_special_tokens(
    {'additional_special_tokens': CERT_SPECIAL_TOKENS}
)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids('<s_cert>')
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

model.to(DEVICE)
print(f'✅ Model loaded  ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)')
print(f'   Special tokens : {len(CERT_SPECIAL_TOKENS)} added')
print(f'   Vocab size     : {len(processor.tokenizer)}')

In [ ]:
import json, os, shutil
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

# Unzip from Drive to local SSD if not already present
LOCAL_DATASET = '/content/dataset_v2'
if not os.path.exists(LOCAL_DATASET):
    print('Unzipping dataset from Drive to local storage ...')
    shutil.unpack_archive(f'{DRIVE_PATH}/dataset_v2_archive.zip', '/content')
    print('Done.')
else:
    print('Dataset already available locally.')

CERT_MAX_LENGTH = 512
FIELD_NAMES = [
    'student_name', 'university_name', 'course_name',
    'specialization', 'pass_class', 'authority_name', 'issue_date',
]


def fields_to_target(fields: dict) -> str:
    inner = ''.join(
        f'<s_{f}>{fields.get(f, "")}</s_{f}>'
        for f in FIELD_NAMES
    )
    return f'<s_cert>{inner}</s_cert>'


def load_metadata(jsonl_path: str) -> list:
    with open(jsonl_path, encoding='utf-8') as fh:
        return [json.loads(ln) for ln in fh if ln.strip()]


class CertificateDataset(Dataset):
    def __init__(self, records: list, dataset_root: str):
        self.records = records
        self.root    = Path(dataset_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = Image.open(self.root / rec['file_name']).convert('RGB')

        pixel_values = processor(image, return_tensors='pt').pixel_values.squeeze()
        target       = fields_to_target(rec)

        input_ids = processor.tokenizer(
            target,
            add_special_tokens=False,
            max_length=CERT_MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze()

        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


print('Loading dataset splits ...')
train_records = load_metadata(f'{LOCAL_DATASET}/metadata_train.jsonl')
val_records   = load_metadata(f'{LOCAL_DATASET}/metadata_val.jsonl')
test_records  = load_metadata(f'{LOCAL_DATASET}/metadata_test.jsonl')

train_ds = CertificateDataset(train_records, LOCAL_DATASET)
val_ds   = CertificateDataset(val_records,   LOCAL_DATASET)
test_ds  = CertificateDataset(test_records,  LOCAL_DATASET)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
print('\nSample target string:')
print(fields_to_target(train_records[0]))

In [ ]:
import os, torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
model.gradient_checkpointing_enable()
torch.cuda.empty_cache()


def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'labels':       torch.stack([b['labels']       for b in batch]),
    }


# ── Training time estimate (free T4, 640×480 images) ──────────────────────────
# Steps per epoch = 2400 train / (bs=1 × accum=8) = 300
# Total steps     = 300 × 4 epochs = 1200
# Speed           ~ 0.12–0.18 it/s  →  ~1.8–2.5 hrs
# With checkpoint every 500 steps you get at least 2 saves before session limit.
# ─────────────────────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_PATH,
    # ── Epochs: 4 is enough; v2 data is more diverse so less memorisation ──
    num_train_epochs=4,
    # ── Batch ─────────────────────────────────────────────────────────────
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,          # effective batch = 8
    per_device_eval_batch_size=1,
    # ── Optimiser ─────────────────────────────────────────────────────────
    optim='adafactor',
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    # ── Memory ────────────────────────────────────────────────────────────
    fp16=True,
    gradient_checkpointing=True,
    # ── Logging ───────────────────────────────────────────────────────────
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    # ── Checkpointing (every 500 steps = ~2 saves on free tier) ──────────
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    # ── Misc ──────────────────────────────────────────────────────────────
    predict_with_generate=False,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

eff_batch        = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch  = len(train_ds) // eff_batch
total_steps      = steps_per_epoch * training_args.num_train_epochs

print('Starting v2 fine-tuning...')
print(f'  Epochs             : {training_args.num_train_epochs}')
print(f'  Steps per epoch    : {steps_per_epoch}')
print(f'  Total steps        : {total_steps}')
print(f'  Effective batch    : {eff_batch}')
print(f'  Image size         : {IMG_H}×{IMG_W}')
print(f'  Checkpoints        → {CKPT_PATH}\n')

trainer.train()

best_ckpt = f'{CKPT_PATH}/best_model'
trainer.save_model(best_ckpt)
processor.save_pretrained(best_ckpt)
print(f'\n✅ Best model saved to {best_ckpt}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RESUME FROM CHECKPOINT (run this cell ONLY if Colab disconnected mid-training)
# ─────────────────────────────────────────────────────────────────────────────
# After Colab reconnects:
#   1. Re-run Section 0 (install packages + mount Drive)
#   2. Re-run Section 3 → cell-s3-load-model-v2  (re-load base model + tokens)
#   3. Re-run Section 3 → cell-s3-dataset-v2    (re-create dataset objects)
#   4. Run THIS cell (resume training from last saved checkpoint)

import os
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Find the latest checkpoint in Drive
ckpt_dirs = sorted(
    [d for d in os.listdir(CKPT_PATH) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[-1])
)

if not ckpt_dirs:
    print('❌ No checkpoints found in Drive. Cannot resume — run from the beginning.')
else:
    resume_ckpt = f'{CKPT_PATH}/{ckpt_dirs[-1]}'
    print(f'Resuming from: {resume_ckpt}')

    trainer.train(resume_from_checkpoint=resume_ckpt)

    best_ckpt = f'{CKPT_PATH}/best_model'
    trainer.save_model(best_ckpt)
    processor.save_pretrained(best_ckpt)
    print(f'\n✅ Training complete. Best model → {best_ckpt}')

## 📊 Section 4 — Evaluation

Metrics:
- **Field-level exact match** (all 7 fields, case-insensitive)
- **Character Error Rate (CER)** for free-text fields
- **Confusion matrix** for `pass_class` (closed-set: 5 classes)
- **Qualitative comparison** — real certificate image vs. extracted JSON


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import re, torch

BEST_MODEL_PATH = f'{CKPT_PATH}/best_model'
print(f'Loading best checkpoint from {BEST_MODEL_PATH} ...')
processor = DonutProcessor.from_pretrained(BEST_MODEL_PATH)
model     = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_PATH)
model.to(DEVICE)
model.eval()
print('✅ Best model loaded.')


def decode_to_fields(token_ids) -> dict:
    """Decode model output token ids → dict of 7 field values."""
    text   = processor.tokenizer.decode(token_ids, skip_special_tokens=False)
    result = {}
    for f in FIELD_NAMES:
        m = re.search(rf'<s_{f}>(.*?)</s_{f}>', text, re.DOTALL)
        result[f] = m.group(1).strip() if m else ''
    return result


def run_inference(image) -> dict:
    """Run the fine-tuned model on a single PIL image."""
    pixel_values  = processor(image.convert('RGB'), return_tensors='pt').pixel_values.to(DEVICE)
    decoder_start = torch.full(
        (1, 1), model.config.decoder_start_token_id, device=DEVICE
    )
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_start,
            max_length=CERT_MAX_LENGTH,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )
    return decode_to_fields(outputs.sequences[0])


print('decode_to_fields and run_inference helpers defined.')

In [ ]:
import editdistance
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
from PIL import Image

FREE_TEXT_FIELDS  = ['student_name', 'university_name', 'course_name', 'authority_name']

EVAL_SUBSET  = min(200, len(test_records))
eval_records = test_records[:EVAL_SUBSET]

exact_matches = defaultdict(int)
cer_totals    = defaultdict(float)
n             = len(eval_records)

pred_pass_classes = []
true_pass_classes = []

for rec in tqdm(eval_records, desc='Evaluating'):
    img  = Image.open(Path(LOCAL_DATASET) / rec['file_name']).convert('RGB')
    pred = run_inference(img)

    for f in FIELD_NAMES:
        if pred.get(f, '').strip().lower() == rec.get(f, '').strip().lower():
            exact_matches[f] += 1

    for f in FREE_TEXT_FIELDS:
        p, t = pred.get(f, ''), rec.get(f, '')
        cer  = editdistance.eval(p, t) / max(len(t), 1)
        cer_totals[f] += cer

    pred_pass_classes.append(pred.get('pass_class', '').strip())
    true_pass_classes.append(rec.get('pass_class', '').strip())

print(f'\n=== Field-level Exact Match (n={n}) ===')
for f in FIELD_NAMES:
    acc = exact_matches[f] / n * 100
    bar = '█' * int(acc / 5)
    print(f'  {f:<22}: {acc:5.1f}%  {bar}')

print(f'\n=== Character Error Rate — free-text fields ===')
for f in FREE_TEXT_FIELDS:
    avg_cer = cer_totals[f] / n
    print(f'  {f:<22}: {avg_cer:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

PASS_CLASSES = ['Distinction', 'First Class', 'Second Class Upper', 'Second Class Lower', 'Pass']

cm   = confusion_matrix(true_pass_classes, pred_pass_classes, labels=PASS_CLASSES)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=PASS_CLASSES)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=30)
plt.title('pass_class Confusion Matrix (v2 model)')
plt.tight_layout()
plt.show()

pc_acc = sum(p == t for p, t in zip(pred_pass_classes, true_pass_classes)) / len(true_pass_classes)
print(f'pass_class accuracy: {pc_acc*100:.1f}%')

In [ ]:
# Qualitative: show 3 test certificates with pred vs. ground truth
import matplotlib.pyplot as plt
import random
from pathlib import Path
from PIL import Image

samples = random.sample(eval_records, min(3, len(eval_records)))

for rec in samples:
    img  = Image.open(Path(LOCAL_DATASET) / rec['file_name']).convert('RGB')
    pred = run_inference(img)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(img)
    axes[0].set_title(rec['file_name'], fontsize=9)
    axes[0].axis('off')

    report = ''
    for f in FIELD_NAMES:
        ok     = '✅' if pred.get(f,'').strip().lower() == rec.get(f,'').strip().lower() else '❌'
        report += f"{ok} {f}\n  GT  : {rec.get(f,'')}\n  Pred: {pred.get(f,'')}\n\n"

    axes[1].text(0.02, 0.98, report, transform=axes[1].transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 🔍 Section 5 — Inference on a Real Certificate

Upload any real certificate image (JPG or PNG) and verify that the v2 model
extracts all 7 fields correctly — even from flowing prose sentences.


In [ ]:
from google.colab import files
from PIL import Image
import io, json

print('Upload a certificate image (JPG or PNG):')
uploaded = files.upload()

fname = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')

print(f'\nImage: {fname}  |  Size: {image.size}')

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 10))
thumb = image.copy()
thumb.thumbnail((500, 700))
plt.imshow(thumb)
plt.axis('off')
plt.title(fname)
plt.show()

In [ ]:
# Full-resolution inference
full_image = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')

result = run_inference(full_image)

# Guarantee all 7 keys are present (schema contract)
OUTPUT_SCHEMA = {
    'student_name': '',
    'university_name': '',
    'course_name': '',
    'specialization': '',
    'pass_class': '',
    'authority_name': '',
    'issue_date': '',
}
output = {**OUTPUT_SCHEMA, **result}

print('=== Extracted Fields ===')
print(json.dumps(output, indent=2, ensure_ascii=False))

assert set(output.keys()) == set(OUTPUT_SCHEMA.keys()), \
    '❌ Schema violation — missing keys!'
print('\n✅ Schema valid — all 7 keys present.')